In [14]:
import pandas as pd

In [15]:
evaluations = pd.read_csv("../data/raw/evaluations.csv")
sessions = pd.read_csv("../data/raw/sessions.csv")
model_pricing = pd.read_csv("../data/raw/model_pricing.csv")
llm_calls = pd.read_csv("../data/raw/llm_calls.csv")

In [16]:
df = llm_calls
df.head()

,call_id,session_id,idempotency_key,model,tool_name,prompt_tokens,completion_tokens,cache_hit,latency_ms,status,logged_cost_usd,called_at,attempt_no
0,CL-0012245,SS800566,IK-0012245,mlg-tutor-lg,explain_grammar,1338,239,True,906,success,0.002523,2026-08-08 22:00:00,1
1,CL-0013004,SS802118,IK-0013004,mlg-asr-align,fetch_audio,1328,413,0,1217,context_overflow,0.000243,1767423822,1
2,CL-0026948,SS801086,IK-0012655,mlg-tutor-lg,none,1298,5,1,1127,success,0.001450,2025-12-25 07:14:46,2
3,CL-0008182,SS800963,IK-0008182,mlg-translate-lg,explain_grammar,1018,67,False,1476,tool_error,0.001029,2026-06-20 11:00:58,1
4,CL-0019992,SS804850,IK-0019992,mlg-tutor-sm,none,20,130,1,293,refusal,0.000108,2025-11-15 07:10:09,1


In [17]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 27854 entries, 0 to 27853
Data columns (total 13 columns):
 #   Column             Non-Null Count  Dtype  
---  ------             --------------  -----  
 0   call_id            27854 non-null  object 
 1   session_id         27854 non-null  object 
 2   idempotency_key    27854 non-null  object 
 3   model              27854 non-null  object 
 4   tool_name          27854 non-null  object 
 5   prompt_tokens      27854 non-null  object 
 6   completion_tokens  27854 non-null  int64  
 7   cache_hit          27854 non-null  object 
 8   latency_ms         27854 non-null  object 
 9   status             27854 non-null  object 
 10  logged_cost_usd    27160 non-null  float64
 11  called_at          27854 non-null  object 
 12  attempt_no         27854 non-null  int64  
dtypes: float64(1), int64(2), object(10)
memory usage: 2.8+ MB


In [18]:
df.isna().sum()

call_id                0
session_id             0
idempotency_key        0
model                  0
tool_name              0
prompt_tokens          0
completion_tokens      0
cache_hit              0
latency_ms             0
status                 0
logged_cost_usd      694
called_at              0
attempt_no             0
dtype: int64

In [19]:
# df.model.value_counts()

# def value_count(column):
#     print(column.value_counts())

# df.drop(columns=["prompt_tokens", "completion_tokens", "logged_cost_usd"]).apply(value_count)

# Fix column data type and value anomalies

In [20]:
numeric_cols = ["prompt_tokens", "completion_tokens", "logged_cost_usd", "attempt_no", "latency_ms"]
string_cols = df.drop(columns=numeric_cols+["called_at", "cache_hit"]).columns.tolist()
boolean_map = {
    True: True,   False:False,
    "True": True, "False":False,
    "TRUE": True,  "FALSE": False,
    "1"   : True,  "0": False,
    "yes" : True,  "no": False,
    1      : True, 0:False
}

def convert_date_time(column):
    try:
        return pd.to_datetime(column, format="mixed")
    except:
        return pd.to_datetime(column,unit="s")
    

def convert_to_int(column):
    #convert all numeric columns to numeric data

    if column.name in numeric_cols:
        #split latency text to remove "ms from it"
        if column.name == "latency_ms":
            column = column.apply(lambda x: x.split(" ")[0])
        if column.name == "prompt_tokens":
            column = column.apply(lambda x: int(x.replace(",","")))
        try:
            return column.apply(lambda x: Decimal(str(x)) if pd.notna(x) else x)
        except:
            return 

    # resolve model naming conflict wiht capitalisation issues , and
    if column.name in string_cols:
        # first bring all model names to lowercase 
        if column.name == "model":
            column = column.apply(lambda x: x.lower())
        return column.astype("string")

    # handle boolean values
    if column.name == "cache_hit":
        return column.map(boolean_map)

    return column


In [21]:
df= df.apply(convert_to_int)
df["called_at"] = df.called_at.apply(convert_date_time)
df.dtypes

C:\Users\chris\AppData\Local\Temp\ipykernel_2988\3562644243.py:16: FutureWarning: The behavior of 'to_datetime' with 'unit' when parsing strings is deprecated. In a future version, strings will be parsed as datetime strings, matching the behavior without a 'unit'. To retain the old behavior, explicitly cast ints or floats to numeric type before calling to_datetime.
  return pd.to_datetime(column,unit="s")


call_id              string[python]
session_id           string[python]
idempotency_key      string[python]
model                string[python]
tool_name            string[python]
prompt_tokens                object
completion_tokens            object
cache_hit                      bool
latency_ms                   object
status               string[python]
logged_cost_usd              object
called_at            datetime64[ns]
attempt_no                   object
dtype: object

In [22]:
df.isna().sum()

call_id                  0
session_id               0
idempotency_key          0
model                    0
tool_name                0
prompt_tokens        27854
completion_tokens    27854
cache_hit                0
latency_ms           27854
status                   0
logged_cost_usd      27854
called_at                0
attempt_no           27854
dtype: int64

In [23]:
df.to_csv("../data/cleaned/llm_calls.csv")

In [24]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 27854 entries, 0 to 27853
Data columns (total 13 columns):
 #   Column             Non-Null Count  Dtype         
---  ------             --------------  -----         
 0   call_id            27854 non-null  string        
 1   session_id         27854 non-null  string        
 2   idempotency_key    27854 non-null  string        
 3   model              27854 non-null  string        
 4   tool_name          27854 non-null  string        
 5   prompt_tokens      0 non-null      object        
 6   completion_tokens  0 non-null      object        
 7   cache_hit          27854 non-null  bool          
 8   latency_ms         0 non-null      object        
 9   status             27854 non-null  string        
 10  logged_cost_usd    0 non-null      object        
 11  called_at          27854 non-null  datetime64[ns]
 12  attempt_no         0 non-null      object        
dtypes: bool(1), datetime64[ns](1), object(5), string(6)
memory us

session table has duplicate columns

In [25]:
df

,call_id,session_id,idempotency_key,model,tool_name,prompt_tokens,completion_tokens,cache_hit,latency_ms,status,logged_cost_usd,called_at,attempt_no
0,CL-0012245,SS800566,IK-0012245,mlg-tutor-lg,explain_grammar,None,None,True,None,success,None,2026-08-08 22:00:00,None
1,CL-0013004,SS802118,IK-0013004,mlg-asr-align,fetch_audio,None,None,False,None,context_overflow,None,2026-01-03 07:03:42,None
2,CL-0026948,SS801086,IK-0012655,mlg-tutor-lg,none,None,None,True,None,success,None,2025-12-25 07:14:46,None
3,CL-0008182,SS800963,IK-0008182,mlg-translate-lg,explain_grammar,None,None,False,None,tool_error,None,2026-06-20 11:00:58,None
4,CL-0019992,SS804850,IK-0019992,mlg-tutor-sm,none,None,None,True,None,refusal,None,2025-11-15 07:10:09,None
...,...,...,...,...,...,...,...,...,...,...,...,...,...
27849,CL-0017641,SS800659,IK-0017641,mlg-asr-align,none,None,None,False,None,success,None,2026-02-21 22:08:24,None
27850,CL-0023967,SS804204,IK-0023967,mlg-tutor-sm,lookup_dictionary,None,None,False,None,success,None,2026-01-30 12:03:06,None
27851,CL-0010532,SS801424,IK-0010532,mlg-translate-sm,none,None,None,False,None,success,None,2026-06-20 09:11:00,None
27852,CL-0006531,SS801501,IK-0006531,mlg-tutor-lg,none,None,None,False,None,timeout,None,2026-06-12 11:00:00,None


In [26]:
from decimal import Decimal

pricing = {
    row["model"]: {
        "prompt": Decimal(str(row["prompt_usd_per_token"])),
        "completion": Decimal(str(row["completion_usd_per_token"]))
    }
    for _, row in model_pricing.iterrows()
}

In [27]:
import sys
sys.path.insert(0, '../..')
from agent_telimetry.costing import calculate_actual_cost

In [ ]:
llm_calls = pd.read_csv("../data/cleaned/llm_calls.csv")
llm_calls.head()

,index,Unnamed: 0,call_id,session_id,idempotency_key,model,tool_name,prompt_tokens,completion_tokens,cache_hit,latency_ms,status,logged_cost_usd,called_at,attempt_no
0,0,0,CL-0012245,SS800566,IK-0012245,mlg-tutor-lg,explain_grammar,NaN,NaN,True,NaN,success,NaN,2026-08-08 22:00:00,NaN
1,1,1,CL-0013004,SS802118,IK-0013004,mlg-asr-align,fetch_audio,NaN,NaN,False,NaN,context_overflow,NaN,2026-01-03 07:03:42,NaN
2,2,2,CL-0026948,SS801086,IK-0012655,mlg-tutor-lg,none,NaN,NaN,True,NaN,success,NaN,2025-12-25 07:14:46,NaN
3,3,3,CL-0008182,SS800963,IK-0008182,mlg-translate-lg,explain_grammar,NaN,NaN,False,NaN,tool_error,NaN,2026-06-20 11:00:58,NaN
4,4,4,CL-0019992,SS804850,IK-0019992,mlg-tutor-sm,none,NaN,NaN,True,NaN,refusal,NaN,2025-11-15 07:10:09,NaN


In [34]:
llm_calls['actual_cost'] = llm_calls.apply(lambda row: calculate_actual_cost(
    model=row["model"],
    completion_tokens=row["completion_tokens"],
    input_tokens=row["prompt_tokens"]
),
axis=1
)

In [35]:
llm_calls.head()

,index,Unnamed: 0,call_id,session_id,idempotency_key,model,tool_name,prompt_tokens,completion_tokens,cache_hit,latency_ms,status,logged_cost_usd,called_at,attempt_no,actual_cost
0,0,0,CL-0012245,SS800566,IK-0012245,mlg-tutor-lg,explain_grammar,NaN,NaN,True,NaN,success,NaN,2026-08-08 22:00:00,NaN,NaN
1,1,1,CL-0013004,SS802118,IK-0013004,mlg-asr-align,fetch_audio,NaN,NaN,False,NaN,context_overflow,NaN,2026-01-03 07:03:42,NaN,NaN
2,2,2,CL-0026948,SS801086,IK-0012655,mlg-tutor-lg,none,NaN,NaN,True,NaN,success,NaN,2025-12-25 07:14:46,NaN,NaN
3,3,3,CL-0008182,SS800963,IK-0008182,mlg-translate-lg,explain_grammar,NaN,NaN,False,NaN,tool_error,NaN,2026-06-20 11:00:58,NaN,NaN
4,4,4,CL-0019992,SS804850,IK-0019992,mlg-tutor-sm,none,NaN,NaN,True,NaN,refusal,NaN,2025-11-15 07:10:09,NaN,NaN
